# 📈 Model Evaluation & Error Analysis - AI LogGuard Phase 3

**Mục đích:** Đánh giá chi tiết performance của ML models và phân tích lỗi

**Nội dung:**
1. Load trained models và test data
2. Detailed metrics analysis
3. Per-class performance breakdown
4. Error analysis (false positives/negatives)
5. Confusion matrix deep dive
6. Feature importance analysis
7. Model comparison
8. Misclassification analysis
9. Confidence calibration
10. Production readiness assessment

## 1. Setup & Load Models

In [ ]:
!pip install scikit-learn joblib pandas matplotlib seaborn numpy xgboost -q

In [ ]:
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
    roc_auc_score,
    average_precision_score,
    cohen_kappa_score,
    matthews_corrcoef
)

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("✅ Libraries imported!")

In [ ]:
# Load models
print("📂 Loading trained models...")
lr_model = joblib.load('models/error_classifier_lr.pkl')
rf_model = joblib.load('models/error_classifier_rf.pkl')
xgb_model = joblib.load('models/error_classifier_xgb.pkl')

# Load test data
X_test = joblib.load('models/X_test.pkl')
y_test = joblib.load('models/y_test.pkl')

# Load metadata
label_encoder = joblib.load('models/label_encoder.pkl')
model_metadata = joblib.load('models/model_metadata.pkl')

print(f"✅ Models loaded!")
print(f"Test samples: {len(y_test)}")
print(f"Classes: {label_encoder.classes_}")

## 2. Generate Predictions

In [ ]:
# Generate predictions for all models
print("🔮 Generating predictions...")

models = {
    'Logistic Regression': lr_model,
    'Random Forest': rf_model,
    'XGBoost': xgb_model
}

predictions = {}
probabilities = {}

for name, model in models.items():
    predictions[name] = model.predict(X_test)
    probabilities[name] = model.predict_proba(X_test)
    print(f"✅ {name} predictions generated")

print("\n✅ All predictions ready!")

## 3. Comprehensive Metrics Comparison

In [ ]:
# Calculate comprehensive metrics
metrics_data = []

for name, y_pred in predictions.items():
    y_proba = probabilities[name]
    
    # Basic metrics
    acc = accuracy_score(y_test, y_pred)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_test, y_pred, average='weighted'
    )
    
    # Advanced metrics
    kappa = cohen_kappa_score(y_test, y_pred)
    mcc = matthews_corrcoef(y_test, y_pred)
    
    metrics_data.append({
        'Model': name,
        'Accuracy': acc,
        'Precision': precision,
        'Recall': recall,
        'F1-Score': f1,
        'Cohen Kappa': kappa,
        'MCC': mcc
    })

metrics_df = pd.DataFrame(metrics_data)
print("\n📊 Comprehensive Metrics Comparison:")
print("=" * 100)
print(metrics_df.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
print("=" * 100)

In [ ]:
# Visualize metrics comparison
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
metrics_to_plot = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'Cohen Kappa', 'MCC']
colors = ['steelblue', 'coral', 'lightgreen', 'gold', 'orchid', 'skyblue']

for idx, (metric, color) in enumerate(zip(metrics_to_plot, colors)):
    ax = axes[idx // 3, idx % 3]
    metrics_df.plot(x='Model', y=metric, kind='bar', ax=ax, legend=False, color=color)
    ax.set_title(f'{metric} Comparison', fontweight='bold')
    ax.set_ylabel(metric)
    ax.set_ylim([0, 1])
    ax.tick_params(axis='x', rotation=45)
    
    # Add values on bars
    for i, v in enumerate(metrics_df[metric]):
        ax.text(i, v + 0.02, f'{v:.3f}', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

# Highlight best model
best_model = metrics_df.loc[metrics_df['F1-Score'].idxmax(), 'Model']
print(f"\n🏆 Best Model (by F1-Score): {best_model}")

## 4. Per-Class Performance Analysis

In [ ]:
# Per-class metrics for best model
best_predictions = predictions[best_model]

print(f"\n📊 Per-Class Performance ({best_model}):")
print("=" * 80)
print(classification_report(
    y_test, 
    best_predictions, 
    target_names=label_encoder.classes_,
    digits=4
))
print("=" * 80)

In [ ]:
# Visualize per-class metrics
precision, recall, f1, support = precision_recall_fscore_support(
    y_test, best_predictions
)

per_class_df = pd.DataFrame({
    'Class': label_encoder.classes_,
    'Precision': precision,
    'Recall': recall,
    'F1-Score': f1,
    'Support': support
})

# Plot
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Metrics comparison
per_class_df.set_index('Class')[['Precision', 'Recall', 'F1-Score']].plot(
    kind='bar', ax=axes[0]
)
axes[0].set_title('Per-Class Metrics', fontweight='bold', fontsize=14)
axes[0].set_ylabel('Score')
axes[0].set_ylim([0, 1])
axes[0].legend(loc='lower right')
axes[0].tick_params(axis='x', rotation=45)
axes[0].grid(axis='y', alpha=0.3)

# Support distribution
per_class_df.plot(x='Class', y='Support', kind='bar', ax=axes[1], legend=False, color='coral')
axes[1].set_title('Test Set Class Distribution', fontweight='bold', fontsize=14)
axes[1].set_ylabel('Number of Samples')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

print("\n📊 Per-Class Metrics Summary:")
print(per_class_df.to_string(index=False))

## 5. Confusion Matrix Deep Dive

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test, best_predictions)
cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Absolute counts
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=label_encoder.classes_,
            yticklabels=label_encoder.classes_,
            ax=axes[0], cbar_kws={'label': 'Count'})
axes[0].set_title('Confusion Matrix (Counts)', fontweight='bold', fontsize=14)
axes[0].set_ylabel('True Label')
axes[0].set_xlabel('Predicted Label')

# Normalized (percentages)
sns.heatmap(cm_normalized, annot=True, fmt='.2%', cmap='Greens',
            xticklabels=label_encoder.classes_,
            yticklabels=label_encoder.classes_,
            ax=axes[1], cbar_kws={'label': 'Percentage'})
axes[1].set_title('Confusion Matrix (Normalized)', fontweight='bold', fontsize=14)
axes[1].set_ylabel('True Label')
axes[1].set_xlabel('Predicted Label')

plt.tight_layout()
plt.show()

In [ ]:
# Identify most confused pairs
print("\n🔍 Most Confused Class Pairs:")
print("=" * 80)

confused_pairs = []
for i in range(len(cm)):
    for j in range(len(cm)):
        if i != j and cm[i, j] > 0:
            confused_pairs.append({
                'True': label_encoder.classes_[i],
                'Predicted': label_encoder.classes_[j],
                'Count': cm[i, j],
                'Percentage': cm_normalized[i, j]
            })

confused_df = pd.DataFrame(confused_pairs).sort_values('Count', ascending=False)
print(confused_df.head(10).to_string(index=False))
print("=" * 80)

## 6. Error Analysis - False Positives & False Negatives

In [ ]:
# Analyze misclassifications
test_df = pd.read_csv('data/synthetic_logs/test.csv')
test_df['predicted_category'] = label_encoder.inverse_transform(best_predictions)
test_df['is_correct'] = (test_df['error_category'] == test_df['predicted_category'])

# Get misclassified samples
misclassified = test_df[~test_df['is_correct']].copy()

print(f"\n❌ Misclassified Samples: {len(misclassified)} / {len(test_df)} ({len(misclassified)/len(test_df)*100:.1f}%)")
print(f"✅ Correctly Classified: {len(test_df) - len(misclassified)} / {len(test_df)} ({(len(test_df)-len(misclassified))/len(test_df)*100:.1f}%)")

In [ ]:
# Show sample misclassifications
print("\n🔍 Sample Misclassifications:")
print("=" * 100)

for idx, row in misclassified.head(10).iterrows():
    print(f"\nFile: {row['file_path']}")
    print(f"Platform: {row['platform']}")
    print(f"True Label: {row['error_category']}")
    print(f"Predicted: {row['predicted_category']}")
    print("-" * 100)

In [ ]:
# Analyze error patterns by platform
error_by_platform = misclassified.groupby('platform').size()
total_by_platform = test_df.groupby('platform').size()
error_rate_by_platform = (error_by_platform / total_by_platform * 100).fillna(0)

print("\n📊 Error Rate by Platform:")
print("=" * 60)
for platform in test_df['platform'].unique():
    err_count = error_by_platform.get(platform, 0)
    total = total_by_platform.get(platform, 0)
    err_rate = error_rate_by_platform.get(platform, 0)
    print(f"{platform:20s}: {err_count}/{total} errors ({err_rate:.1f}%)")
print("=" * 60)

## 7. Confidence Analysis

In [ ]:
# Get confidence scores (max probability)
best_proba = probabilities[best_model]
confidence_scores = np.max(best_proba, axis=1)

test_df['confidence'] = confidence_scores

# Analyze confidence distribution
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Overall confidence distribution
axes[0].hist(confidence_scores, bins=20, color='steelblue', edgecolor='black', alpha=0.7)
axes[0].axvline(np.mean(confidence_scores), color='red', linestyle='--', 
                linewidth=2, label=f'Mean: {np.mean(confidence_scores):.3f}')
axes[0].set_xlabel('Confidence Score')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Confidence Score Distribution', fontweight='bold')
axes[0].legend()

# Confidence by correctness
correct_conf = test_df[test_df['is_correct']]['confidence']
incorrect_conf = test_df[~test_df['is_correct']]['confidence']

axes[1].hist([correct_conf, incorrect_conf], bins=15, 
             label=['Correct', 'Incorrect'], 
             color=['green', 'red'], alpha=0.6, edgecolor='black')
axes[1].set_xlabel('Confidence Score')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Confidence: Correct vs Incorrect Predictions', fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"\n📊 Confidence Statistics:")
print(f"Mean Confidence (Correct): {correct_conf.mean():.4f}")
print(f"Mean Confidence (Incorrect): {incorrect_conf.mean():.4f}")
print(f"Median Confidence: {np.median(confidence_scores):.4f}")

In [ ]:
# Confidence-based accuracy
confidence_bins = [0, 0.5, 0.7, 0.8, 0.9, 1.0]
bin_labels = ['<0.5', '0.5-0.7', '0.7-0.8', '0.8-0.9', '0.9-1.0']

test_df['confidence_bin'] = pd.cut(test_df['confidence'], bins=confidence_bins, labels=bin_labels)

acc_by_confidence = test_df.groupby('confidence_bin')['is_correct'].agg(['mean', 'count'])
acc_by_confidence.columns = ['Accuracy', 'Count']

print("\n📊 Accuracy by Confidence Level:")
print("=" * 60)
print(acc_by_confidence.to_string())
print("=" * 60)

# Plot
fig, ax = plt.subplots(figsize=(10, 6))
acc_by_confidence['Accuracy'].plot(kind='bar', ax=ax, color='teal')
ax.set_title('Accuracy by Confidence Level', fontweight='bold', fontsize=14)
ax.set_xlabel('Confidence Range')
ax.set_ylabel('Accuracy')
ax.set_ylim([0, 1])
ax.tick_params(axis='x', rotation=0)

# Add count labels
for i, (acc, count) in enumerate(zip(acc_by_confidence['Accuracy'], acc_by_confidence['Count'])):
    ax.text(i, acc + 0.02, f'{acc:.2%}\n(n={int(count)})', 
            ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

## 8. Feature Importance Analysis

In [ ]:
# Get feature importances from tree-based models
feature_info = joblib.load('models/feature_info.pkl')
tfidf = joblib.load('models/tfidf_vectorizer.pkl')

# Reconstruct feature names
tfidf_features = tfidf.get_feature_names_out().tolist()
structural_features = feature_info['structural_feature_names']
platform_features = feature_info['platform_feature_names']
all_feature_names = tfidf_features + structural_features + platform_features

print(f"Total features: {len(all_feature_names)}")
print(f"  - TF-IDF: {len(tfidf_features)}")
print(f"  - Structural: {len(structural_features)}")
print(f"  - Platform: {len(platform_features)}")

In [ ]:
# Compare feature importances across models
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Random Forest
rf_importance = rf_model.feature_importances_
top_n = 20
rf_top_indices = np.argsort(rf_importance)[-top_n:][::-1]
rf_top_names = [all_feature_names[i] for i in rf_top_indices]
rf_top_scores = rf_importance[rf_top_indices]

axes[0].barh(range(top_n), rf_top_scores, color='forestgreen')
axes[0].set_yticks(range(top_n))
axes[0].set_yticklabels(rf_top_names)
axes[0].set_xlabel('Importance Score')
axes[0].set_title(f'Top {top_n} Features - Random Forest', fontweight='bold')
axes[0].invert_yaxis()

# XGBoost
xgb_importance = xgb_model.feature_importances_
xgb_top_indices = np.argsort(xgb_importance)[-top_n:][::-1]
xgb_top_names = [all_feature_names[i] for i in xgb_top_indices]
xgb_top_scores = xgb_importance[xgb_top_indices]

axes[1].barh(range(top_n), xgb_top_scores, color='darkorange')
axes[1].set_yticks(range(top_n))
axes[1].set_yticklabels(xgb_top_names)
axes[1].set_xlabel('Importance Score')
axes[1].set_title(f'Top {top_n} Features - XGBoost', fontweight='bold')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

In [ ]:
# Analyze feature type importance
def categorize_feature_importance(importances):
    tfidf_importance = np.sum(importances[:len(tfidf_features)])
    structural_importance = np.sum(importances[len(tfidf_features):len(tfidf_features)+len(structural_features)])
    platform_importance = np.sum(importances[len(tfidf_features)+len(structural_features):])
    
    return {
        'TF-IDF': tfidf_importance,
        'Structural': structural_importance,
        'Platform': platform_importance
    }

rf_category_importance = categorize_feature_importance(rf_importance)
xgb_category_importance = categorize_feature_importance(xgb_importance)

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Random Forest
axes[0].pie(rf_category_importance.values(), 
            labels=rf_category_importance.keys(),
            autopct='%1.1f%%',
            startangle=90,
            colors=['skyblue', 'lightcoral', 'lightgreen'])
axes[0].set_title('Feature Type Importance - Random Forest', fontweight='bold')

# XGBoost
axes[1].pie(xgb_category_importance.values(),
            labels=xgb_category_importance.keys(),
            autopct='%1.1f%%',
            startangle=90,
            colors=['skyblue', 'lightcoral', 'lightgreen'])
axes[1].set_title('Feature Type Importance - XGBoost', fontweight='bold')

plt.tight_layout()
plt.show()

## 9. Low Confidence Predictions Analysis

In [ ]:
# Identify low confidence predictions
low_confidence_threshold = 0.6
low_conf_samples = test_df[test_df['confidence'] < low_confidence_threshold].copy()

print(f"\n⚠️  Low Confidence Predictions (<{low_confidence_threshold}): {len(low_conf_samples)} samples")
print(f"Accuracy on low confidence: {low_conf_samples['is_correct'].mean():.2%}")

if len(low_conf_samples) > 0:
    print("\n🔍 Sample Low Confidence Cases:")
    print("=" * 100)
    for idx, row in low_conf_samples.head(5).iterrows():
        print(f"\nFile: {row['file_path']}")
        print(f"True: {row['error_category']}")
        print(f"Predicted: {row['predicted_category']}")
        print(f"Confidence: {row['confidence']:.3f}")
        print(f"Correct: {'✅' if row['is_correct'] else '❌'}")
        print("-" * 100)

## 10. Production Readiness Assessment

In [ ]:
# Production readiness checklist
print("\n" + "=" * 80)
print("🚀 PRODUCTION READINESS ASSESSMENT")
print("=" * 80)

# Calculate key metrics
best_acc = metrics_df.loc[metrics_df['Model'] == best_model, 'Accuracy'].values[0]
best_f1 = metrics_df.loc[metrics_df['Model'] == best_model, 'F1-Score'].values[0]
avg_confidence = confidence_scores.mean()
low_conf_pct = (confidence_scores < 0.6).mean() * 100

# Checklist
checklist = [
    ("Model Accuracy > 70%", best_acc > 0.70, f"{best_acc:.1%}"),
    ("F1-Score > 0.70", best_f1 > 0.70, f"{best_f1:.3f}"),
    ("Average Confidence > 0.75", avg_confidence > 0.75, f"{avg_confidence:.3f}"),
    ("Low Confidence Cases < 20%", low_conf_pct < 20, f"{low_conf_pct:.1f}%"),
    ("All classes have F1 > 0.60", per_class_df['F1-Score'].min() > 0.60, f"Min: {per_class_df['F1-Score'].min():.3f}"),
]

passed = 0
for criterion, status, value in checklist:
    icon = "✅" if status else "❌"
    passed += int(status)
    print(f"{icon} {criterion:40s} [{value}]")

print("\n" + "=" * 80)
print(f"Overall Score: {passed}/{len(checklist)} ({passed/len(checklist)*100:.0f}%)")

if passed >= len(checklist) * 0.8:
    print("\n🎉 Model is READY for production!")
elif passed >= len(checklist) * 0.6:
    print("\n⚠️  Model is ACCEPTABLE but needs improvement")
else:
    print("\n❌ Model NOT READY - requires significant improvement")

print("=" * 80)

## 11. Recommendations

In [ ]:
print("\n" + "=" * 80)
print("💡 RECOMMENDATIONS")
print("=" * 80)

recommendations = []

# Check for class imbalance issues
if per_class_df['F1-Score'].std() > 0.15:
    recommendations.append(
        "⚠️  High variance in per-class F1-scores detected\n"
        "   → Consider collecting more data for underperforming classes\n"
        "   → Try SMOTE or other oversampling techniques"
    )

# Check confidence distribution
if low_conf_pct > 15:
    recommendations.append(
        f"⚠️  {low_conf_pct:.1f}% of predictions have confidence <0.6\n"
        "   → Implement confidence-based routing to LLM for verification\n"
        "   → Consider ensemble methods"
    )

# Check misclassification patterns
if len(confused_df) > 0:
    top_confusion = confused_df.iloc[0]
    if top_confusion['Count'] > 3:
        recommendations.append(
            f"⚠️  High confusion between '{top_confusion['True']}' and '{top_confusion['Predicted']}'\n"
            f"   → These classes may have overlapping patterns\n"
            f"   → Review feature engineering for these specific classes"
        )

# General recommendations
recommendations.append(
    "✅ Implement hybrid ML + LLM pipeline:\n"
    "   → Use ML for fast classification\n"
    "   → Use LLM for detailed explanation and fix suggestions\n"
    "   → Route low-confidence predictions to LLM for verification"
)

recommendations.append(
    "✅ Set up continuous learning pipeline:\n"
    "   → Collect user feedback on predictions\n"
    "   → Retrain model periodically with verified corrections\n"
    "   → Monitor model drift"
)

for i, rec in enumerate(recommendations, 1):
    print(f"\n{i}. {rec}")

print("\n" + "=" * 80)

## ✅ Summary

**Evaluation Complete!**

Key Findings:
- ✅ Comprehensive metrics calculated for all models
- ✅ Per-class performance analyzed
- ✅ Confusion patterns identified
- ✅ Confidence calibration assessed
- ✅ Feature importance analyzed
- ✅ Production readiness evaluated
- ✅ Actionable recommendations provided

**Next Steps:**
➡️ Notebook 05: Production Integration & Export (integrate model into CLI)